### Collect Events
Used for mapping category i.e. sports, econ, politics

In [11]:
import requests
import pandas as pd
import time
from utils import clean_text_cols, clean_datetime_cols

url_type = 'events'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_events_data = []
cursor = None
max_pages = 10 

print(f"Fetching all {url_type} data from Kalshi...")

for page in range(max_pages):
    params = {"status": "open", "limit": 200, "cursor": cursor}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_events_data.extend(batch)
    
    cursor = data.get('cursor')
    print(f"Page {page+1} fetched. Total events: {len(all_events_data)}")
    
    if not cursor:
        break
    time.sleep(0.5)

events_df = pd.DataFrame(all_events_data)

# Mapping & Filtering
event_col_map = {
    'series_ticker': 'series_id',
    'event_ticker': 'event_id',
    'category': 'tag',
    'title': 'event_title',
    'sub_title': 'event_sub_title'
}
events_df = events_df[[c for c in event_col_map.keys() if c in events_df.columns]].rename(columns=event_col_map)


Fetching all events data from Kalshi...
Page 1 fetched. Total events: 200
Page 2 fetched. Total events: 400
Page 3 fetched. Total events: 600
Page 4 fetched. Total events: 800
Page 5 fetched. Total events: 1000
Page 6 fetched. Total events: 1200
Page 7 fetched. Total events: 1400
Page 8 fetched. Total events: 1600
Page 9 fetched. Total events: 1800
Page 10 fetched. Total events: 2000


### Collect Markets
This is the pricing data

In [12]:
url_type = 'markets'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_market_data = []
cursor = None
max_pages = 5 

print("Fetching markets from Kalshi...")
for page in range(max_pages):
    params = {'mve_filter': 'exclude', 'status': 'open', 'limit': 1000, 'cursor': cursor}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_market_data.extend(batch)
    
    cursor = data.get('cursor')
    print(f"Page {page+1} complete. Total records: {len(all_market_data)}")
    
    if not cursor:
        break
    time.sleep(0.5)

market_df = pd.DataFrame(all_market_data)
market_df['description'] = market_df['rules_primary'] + market_df['rules_secondary'] 

market_col_map = {
    'ticker': 'market_id',
    'event_ticker': 'event_id',
    'title': 'event_title',
    'subtitle': 'event_sub_title',
    'description': 'description',
    'yes_ask_dollars': 'yes_ask',
    'yes_bid_dollars': 'yes_bid',
    'no_ask_dollars': 'no_ask',
    'no_bid_dollars': 'no_bid',
    'close_time': 'close_time',
    'expiration_time': 'expiration'
}

market_df = market_df[[c for c in market_col_map.keys() if c in market_df.columns]].rename(columns=market_col_map)

Fetching markets from Kalshi...
Page 1 complete. Total records: 1000
Page 2 complete. Total records: 2000
Page 3 complete. Total records: 3000
Page 4 complete. Total records: 4000
Page 5 complete. Total records: 5000


### Backfill & Merge
Ensure our events, and market match

In [13]:
all_market_tickers = market_df['event_id'].unique()
known_event_tickers = events_df['event_id'].unique()
missing_tickers = [t for t in all_market_tickers if t not in known_event_tickers]

backfilled_events = []
if missing_tickers:
    print(f"Backfilling {len(missing_tickers)} missing events...")
    for ticker in missing_tickers:
        res = requests.get(f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}")
        if res.status_code == 200:
            backfilled_events.append(res.json().get('event', {}))
        time.sleep(0.05) 

if backfilled_events:
    new_events_df = pd.DataFrame(backfilled_events)
    new_events_df = new_events_df[[c for c in event_col_map.keys() if c in new_events_df.columns]].rename(columns=event_col_map)
    full_events_df = pd.concat([events_df, new_events_df], ignore_index=True)
else:
    full_events_df = events_df

cols_to_use = full_events_df.columns.difference(market_df.columns).tolist()
cols_to_use.append('event_id')

# 2. Merge using only the unique columns from the events dataframe
kalshi_master_df = pd.merge(
    market_df, 
    full_events_df[cols_to_use], 
    on='event_id', 
    how='inner'
)

Backfilling 413 missing events...


In [15]:
# Clean using utils
kalshi_master_df = clean_text_cols(kalshi_master_df)
kalshi_master_df = clean_datetime_cols(kalshi_master_df, date_cols=['close_time', 'expiration'])
kalshi_master_df.to_csv('data/markets/kalshi_markets.csv', index=False)

kalshi_events_df = clean_text_cols(full_events_df)
kalshi_events_df.to_csv('data/events/kalshi_events.csv', index=False)